A website chatbot that can answer questions about the website's content and provide assistance to users. The chatbot will be integrated into the website's interface and will use natural language processing to understand user queries and provide relevant responses. It will also have the ability to learn from user interactions and improve its responses over time.

In [1]:
%pip install -q -U langchain langchain-community langchain-openai langchain-huggingface langchain-text-splitters langchain-classic langchain-pinecone transformers pinecone huggingface_hub unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.3 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 127.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 55.4 MB/s et

In [3]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_openai import ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from langchain_classic.chains import RetrievalQAWithSourcesChain
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import notebook_login
import textwrap
import sys
import os
import torch

In [4]:
import nltk
nltk.download('punkt')
nltk.download('averged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Error loading averged_perceptron_tagger: Package
[nltk_data]     'averged_perceptron_tagger' not found in index


False

In [5]:
URLs = [
    "https://docs.python.org/3/tutorial/index.html",
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Vector_database",
]

In [6]:
loaders=UnstructuredURLLoader(urls=URLs)
data=loaders.load()

In [7]:
data

[Document(metadata={'source': 'https://docs.python.org/3/tutorial/index.html'}, page_content='Navigation\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\n\nPython logo\n\nPython »\n\n3.14.3 Documentation »\n\nThe Python Tutorial\n\n|\n\nThe Python Tutorial¶\n\nTip\n\nThis tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.\n\nPython is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.\n\nThe Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributi

In [8]:
for i, doc in enumerate(data, start=1):
    source = doc.metadata.get("source", "unknown")
    print(f"\n{'='*80}")
    print(f"Document {i} | Source: {source}")
    print(f"{'='*80}\n")
    print(doc.page_content)


Document 1 | Source: https://docs.python.org/3/tutorial/index.html

Navigation

index

modules |

next |

previous |

Python logo

Python »

3.14.3 Documentation »

The Python Tutorial

|

The Python Tutorial¶

Tip

This tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.

Python is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.

The Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributions of and pointers to many free third party Python m

In [9]:
len(data)

3

In [10]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(data)

In [11]:
texts

[Document(metadata={'source': 'https://docs.python.org/3/tutorial/index.html'}, page_content='Navigation\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\n\nPython logo\n\nPython »\n\n3.14.3 Documentation »\n\nThe Python Tutorial\n\n|\n\nThe Python Tutorial¶\n\nTip\n\nThis tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.\n\nPython is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.\n\nThe Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributi

In [12]:
embeddings = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
query_result=embeddings.embed_query("hello world")
len(query_result)

768

In [14]:
query_result

[0.026249706745147705,
 0.013395607471466064,
 -0.004533133003860712,
 -0.021791422739624977,
 0.05455183610320091,
 -0.004966470878571272,
 0.006655557081103325,
 0.03062625601887703,
 -0.005762824323028326,
 -0.004562034271657467,
 -0.003313301596790552,
 -0.0484962984919548,
 -0.011364065110683441,
 0.03507744148373604,
 0.09309469908475876,
 -0.0866873636841774,
 0.051086537539958954,
 0.009886160492897034,
 -0.06356934458017349,
 -0.008550204336643219,
 0.007054392248392105,
 -0.003862351179122925,
 0.024744288995862007,
 0.04288491606712341,
 0.035094160586595535,
 -0.02984827198088169,
 0.010252618230879307,
 0.02234490029513836,
 0.020889993757009506,
 0.00949221383780241,
 -0.03304434195160866,
 -0.012284105643630028,
 0.05352889373898506,
 0.02542915940284729,
 2.022177341132192e-06,
 -0.034191031008958817,
 0.009610011242330074,
 -0.016484549269080162,
 0.005609536077827215,
 -0.004250032361596823,
 -0.022801222279667854,
 0.04035475477576256,
 0.0030519880820065737,
 0.0313

In [15]:
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY', 'pcsk_oVVak_5tFrY3VzTVtpgdeVBgciEYxmJEM6451cuo9e9TejF4ZK5VLFw6QkUdDVFTcyLTR')
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY

In [16]:

pc = Pinecone(api_key=PINECONE_API_KEY)

In [17]:
index_name = "llama2"

In [18]:
vectorstore = PineconeVectorStore.from_texts(
    [t.page_content for t in texts],
    embedding=embeddings,
    index_name=index_name
)

In [4]:
model="mistralai/Mistral-7B-Instruct-v0.2"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model)
llm_model = AutoModelForCausalLM.from_pretrained(
    model,
    device_map='auto',
    torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
pipe=pipeline("text-generation",
              model=llm_model,
              tokenizer=tokenizer,
              torch_dtype=torch.bfloat16,
              device_map='auto',
              max_new_tokens=128,
              top_k=30,
              num_return_sequences=1,
              eos_token_id=tokenizer.eos_token_id)

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'max_new_tokens', 'num_return_sequences', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
llm=HuggingFacePipeline(pipeline=pipe , model_kwrargs=("temperature": 0.7))